In [15]:
import torch, torch.nn as nn

check_point = torch.load(
    "model_and_optimizer.pth",
    map_location=torch.device("mps"),
    weights_only=True,
)


In [16]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, num_heads, context_length, qkv_bias=False, dropout=0.5):
        super().__init__()

        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        self.d_in = d_in
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.Wq = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.Wk = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.Wv = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(p=dropout)
        self.out_proj = nn.Linear(d_out, d_out)
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        batch_size, number_of_tokens, d_in = x.shape # (b, t, c)

        queries = self.Wq(x)
        keys = self.Wk(x)
        values = self.Wv(x)

        queries = queries.view(batch_size, number_of_tokens, self.num_heads, self.head_dim) # [b,t,c] -> [b,t,nh,hd]
        keys = keys.view(batch_size, number_of_tokens, self.num_heads, self.head_dim) # [b,t,c] -> [b,t,nh,hd]
        values = values.view(batch_size, number_of_tokens, self.num_heads, self.head_dim) # [b,t,c] -> [b,t,nh,hd]

        queries = queries.transpose(1, 2) # [b,t,nh,hd] -> [b,nh,t,hd]
        keys = keys.transpose(1, 2) # [b,t,nh,hd] -> [b,nh,t,hd]
        values = values.transpose(1, 2) # [b,t,nh,hd] -> [b,nh,t,hd]

        attention_scores = queries @ keys.transpose(-2, -1) # [b,nh,t,hd] @ [b,nh,hd,t] -> [b,nh,t,t]

        attention_scores.masked_fill_(self.mask.bool()[:number_of_tokens, :number_of_tokens], float('-inf'))

        attention_weights = torch.softmax(attention_scores/ keys.shape[-1]**0.5, dim=-1)

        attention_weights = self.dropout(attention_weights)

        context_vector = attention_weights @ values

        context_vector = context_vector.transpose(1, 2) # [b,nh,t,hd] -> [b,t,nh,hd]
        context_vector = context_vector.contiguous().view(batch_size, number_of_tokens, self.d_out) # [b,t,nh,hd] -> [b,t,c]

        context_vector = self.out_proj(context_vector)

        return context_vector
        
class LayerNorm(nn.Module):
    def __init__(self, embedding_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(embedding_dim))
        self.shift = nn.Parameter(torch.zeros(embedding_dim))
        
    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        variance = x.var(dim=-1, keepdim=True, unbiased=False)
        normalized_x = (x - mean) / torch.sqrt(variance + self.eps)
        return self.scale * normalized_x + self.shift

class GeLU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(torch.sqrt(torch.tensor(2.0 / torch.pi)) * (x + 0.044715 * torch.pow(x, 3))))

class FeedForward(nn.Module):
    def __init__(self, embedding_dim):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(embedding_dim, 4 * embedding_dim),
            nn.GELU(),
            nn.Linear(4 * embedding_dim, embedding_dim)
        )
        
    def forward(self, x):
        return self.layers(x)    

class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.mha = MultiHeadAttention(
            d_in=config["embedding_dim"],
            d_out=config["embedding_dim"],
            num_heads=config["n_heads"],
            context_length=config["context_length"],
            qkv_bias=config["qkv_bias"],
            dropout=config["drop_rate"]
        )
        self.ln1 = LayerNorm(config["embedding_dim"])
        self.ln2 = LayerNorm(config["embedding_dim"])
        self.ffn = FeedForward(config["embedding_dim"])
        self.dropout = nn.Dropout(config["drop_rate"])

    def forward(self, x):

        #component 1: Multi-head attention
        ln1_out = self.ln1(x)
        mha_out = self.mha(ln1_out)
        dropout_1_out = self.dropout(mha_out)
        x = x + dropout_1_out # skip connection

        #component 2: Feed forward network
        ln2_out = self.ln2(x)
        ffn_out = self.ffn(ln2_out)
        dropout_2_out = self.dropout(ffn_out)
        x = x + dropout_2_out # skip connection

        return x
        
class GPTModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.token_embedding = nn.Embedding(config["vocab_size"], config["embedding_dim"])
        self.positional_embedding = nn.Embedding(config["context_length"], config["embedding_dim"])
        self.dropout = nn.Dropout(config["drop_rate"])
        self.transformer_blocks = nn.Sequential(*[
            TransformerBlock(config) for _ in range(config["n_layers"])
        ])
        self.final_layer_norm = LayerNorm(config["embedding_dim"])
        self.out_head = nn.Linear(config["embedding_dim"], config["vocab_size"])
        
    def forward(self, x):
        batch_size, number_of_tokens = x.shape
        x = self.token_embedding(x) + self.positional_embedding(torch.arange(number_of_tokens, device=x.device))
        x = self.dropout(x)
        x = self.transformer_blocks(x)
        x = self.final_layer_norm(x)
        logits = self.out_head(x)
        return logits

In [17]:
GPT_CONFIG = {
    "vocab_size": 50257,
    "context_length": 256,
    "embedding_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False,
    
}

In [18]:
model = GPTModel(GPT_CONFIG)
model.load_state_dict(check_point["model_state_dict"])
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.0001,
    weight_decay=0.01,
)
optimizer.load_state_dict(check_point["optimizer_state_dict"])


model.eval()



GPTModel(
  (token_embedding): Embedding(50257, 768)
  (positional_embedding): Embedding(256, 768)
  (dropout): Dropout(p=0.1, inplace=False)
  (transformer_blocks): Sequential(
    (0): TransformerBlock(
      (mha): MultiHeadAttention(
        (Wq): Linear(in_features=768, out_features=768, bias=False)
        (Wk): Linear(in_features=768, out_features=768, bias=False)
        (Wv): Linear(in_features=768, out_features=768, bias=False)
        (dropout): Dropout(p=0.1, inplace=False)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
      )
      (ln1): LayerNorm()
      (ln2): LayerNorm()
      (ffn): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (mha): MultiHeadAttention(
        (Wq):

## What was saved

The file `model_and_optimizer.pth` is a dictionary with two entries.

### `model_state_dict`

These are the model weights: 174 tensors. Twelve of them are the causal `mask` buffers, shape `[256, 256]`, which were registered with each attention block and are not trained. The other 162 tensors are the parameters. Their elements add up to **162,469,969**.

`Wq`, `Wk`, and `Wv` are saved. `qkv_bias` was `False`, so each has a weight and no bias. There is one set per block, blocks `0` through `11`.

| name | shape | count | parameters |
|---|---|---|---|
| `token_embedding.weight` | `[50257, 768]` | 1 | 38,597,376 |
| `positional_embedding.weight` | `[256, 768]` | 1 | 196,608 |
| `transformer_blocks.N.mha.Wq.weight` | `[768, 768]` | 12 | 7,077,888 |
| `transformer_blocks.N.mha.Wk.weight` | `[768, 768]` | 12 | 7,077,888 |
| `transformer_blocks.N.mha.Wv.weight` | `[768, 768]` | 12 | 7,077,888 |
| `transformer_blocks.N.mha.out_proj.weight` | `[768, 768]` | 12 | 7,077,888 |
| `transformer_blocks.N.mha.out_proj.bias` | `[768]` | 12 | 9,216 |
| `transformer_blocks.N.ln1.scale`, `.shift` | `[768]` | 12 each | 18,432 |
| `transformer_blocks.N.ln2.scale`, `.shift` | `[768]` | 12 each | 18,432 |
| `ffn` expand weight, bias | `[3072, 768]`, `[3072]` | 12 each | 28,348,416 |
| `ffn` contract weight, bias | `[768, 3072]`, `[768]` | 12 each | 28,320,768 |
| `final_layer_norm.scale`, `.shift` | `[768]` | 1 each | 1,536 |
| `out_head.weight` | `[50257, 768]` | 1 | 38,597,376 |
| `out_head.bias` | `[50257]` | 1 | 50,257 |
| **total** | | **162 tensors** | **162,469,969** |

`out_proj` is the linear layer that mixes the 12 attention heads back into one 768-wide vector. The feed-forward layer in each block expands 768 to 3072 and contracts back to 768.

Generation only needs this entry:

```python
model.load_state_dict(check_point["model_state_dict"])
model.eval()
```

### `optimizer_state_dict`

This is AdamW's memory of training, not part of the network. A forward pass never reads it.

For every one of the 162,469,969 parameters, Adam stores two more tensors of the same shape: a momentum average (`exp_avg`) and a squared-gradient average (`exp_avg_sq`). It also stores a step counter per parameter. That is 486 extra tensors, 324,940,100 numbers, plus the settings used during training (`lr`, `betas`, `weight_decay`).

Load it only to continue training. Create the optimizer on `model.parameters()` first, then copy this state in. `load_state_dict` on the optimizer replaces the `lr` and `weight_decay` you just passed with the values that were saved.


In [35]:
import tiktoken, torch

def text_to_token_ids(texts, tokenizer):
    encoded = tokenizer.encode(texts, allowed_special={"<|endoftext|>"})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0) 
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0)
    return tokenizer.decode(flat.tolist())
    
def generate_text(top_k, temperature, model, max_new_tokens, start_context, tokenizer, eos_id = None, device = 'cpu' ):
    model.eval()
    context_size = model.positional_embedding.weight.shape[0]
    token_ids = text_to_token_ids(start_context, tokenizer).to(device)
    for _ in range(max_new_tokens):
        with torch.no_grad():
            idx_cond = token_ids[:, -context_size:]
            logits = model(idx_cond)
            logits = logits[:, -1, :]
            if top_k is not None:
                top_k_logits, top_k_positions = torch.topk(logits, top_k)
                masked_logits = torch.where(condition = logits < top_k_logits[0][-1], input = torch.tensor(float('-inf'), device = device), other = logits)
            if temperature > 0:
                masked_logits_temperature_scaled = torch.softmax(masked_logits/temperature, dim = -1)
            else:
                masked_logits_temperature_scaled = torch.softmax(masked_logits, dim = -1)
            next_token = torch.multinomial(masked_logits_temperature_scaled, num_samples = 1)

            if next_token is not eos_id:
                token_ids = torch.cat((token_ids, next_token), dim=-1)

    return token_ids_to_text(token_ids, tokenizer)

In [20]:
tokenizer =  tiktoken.get_encoding("gpt2")

generated_text =  generate_text(
    top_k = 25,
    temperature = 0.5,
    model = model,
    max_new_tokens = 200, 
    start_context = "Every effort moves you",
    tokenizer = tokenizer, 

)
print(f"Output text: \n {generated_text}")

Output text: 
 Every effort moves you?"

"Yes--quite insensible to the irony. She wanted him vindicated--and by me!"

He laughed again, and threw back his head to look up at the sketch of the donkey. "There were days when I couldn't look at that thing--couldn't face it. But I forced myself to put it here; and now it's cured me--cured me. That's the reason why I don't dabble any more, my dear Rickham; or rather Stroud himself is the reason."

For the first time my idle curiosity about my companion turned into a serious desire to understand him better.

"I wish you'd tell me how it happened," I said.

He stood looking up at the sketch, and twirling between his fingers a cigarette he had forgotten to light. Suddenly he turned toward me.

"I'd rather like to tell you--because I've always suspected you of loathing my work


# Training the loaded model

In [36]:
from torch.utils.data import Dataset, DataLoader

class TrainingDataset(Dataset):
    def __init__(self, text, tokenizer, context_length, stride):
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(text)
        for i in range(0, len(token_ids) - context_length, stride):
            input_ids = token_ids[i:i+context_length]
            target_ids = token_ids[i+1:i+context_length+1]
            self.input_ids.append(torch.tensor(input_ids))
            self.target_ids.append(torch.tensor(target_ids))

    def __len__(self):
        return len(self.input_ids)
            
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

def create_dataloader(data, tokenizer, max_length=256, stride=128, batch_size=4, shuffle=True, drop_last=True, num_workers=0):
    dataset = TrainingDataset(data, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers,
    )
    return dataloader

file_path  = "the-verdict.txt"
with open(file_path, "r") as f:
    text_data = f.read()
    
train_ratio = 0.9
split_index = int(len(text_data) * train_ratio)   
train_data = text_data[:split_index]
val_data = text_data[split_index:]

train_loader = create_dataloader(
    train_data,
    tokenizer,
    max_length=GPT_CONFIG["context_length"],
    stride=GPT_CONFIG["context_length"],
    batch_size=2,
    shuffle=True,
    drop_last=True,
    num_workers=0
)

val_loader = create_dataloader(
    val_data,
    tokenizer,
    max_length=GPT_CONFIG["context_length"],
    stride=GPT_CONFIG["context_length"],
    batch_size=2,
    shuffle=True,
    drop_last=True,
    num_workers=0
)

In [37]:
def calculate_loss_batch(loader, model, device, num_batches = None):
    total_loss = 0 

    if len(loader) == 0:
        return ('nan')
    elif num_batches is not None:
        num_batches = min(num_batches, len(loader))

    for batch_idx, (input_batch, target_batch) in enumerate(loader):
        
        if batch_idx < num_batches:
            input_batch = input_batch.to(device)
            target_batch = target_batch.to(device)

            logits = model(input_batch)
            batch_loss = nn.functional.cross_entropy(logits.view(-1, logits.shape[-1]), target_batch.view(-1))
            total_loss += batch_loss.item()

        else:
            break

    return total_loss / num_batches

def evaluate_model(train_loader, val_loader, model, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss  =  calculate_loss_batch(train_loader, model, device, eval_iter)
        val_loss = calculate_loss_batch(val_loader, model, device, eval_iter)
    return train_loss, val_loss


def train_model(model, optimizer, train_loader, val_loader, num_epochs, device, eval_freq, eval_iter, start_context, tokenizer):
    model.train()
    for state in optimizer.state.values():
        for key, value in state.items():
            if torch.is_tensor(value):
                state[key] = value.to(device)
    model.to(device)
    train_losses = []
    val_losses = []
    track_tokens_seen = []
    global_step = 0
    tokens_seen = 0

    # step 1: iterate over epochs
    for epoch in range(num_epochs):
        # step 2: iterate over batches in each training epoch
        for input_batch, target_batch in train_loader:

            # save model to device
            input_batch = input_batch.to(device)
            target_batch = target_batch.to(device)

            # reset gradients
            # step 3: reset loss gradients from previous batch iteration
            optimizer.zero_grad()

            # forward pass
            # step 4: calculate loss on the batch
            logits =  model(input_batch)
            loss = nn.functional.cross_entropy(logits.view(-1, logits.shape[-1]), target_batch.view(-1))

            # backward pass
            # step 5: backward pass to calculate loss gradients
            loss.backward()

            # update weights
            # step 6: update model weights using loss gradients
            optimizer.step()

            global_step += 1
            tokens_seen += input_batch.numel()

            # evaluate model performance
            # step 7: prin training and val loss
            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(train_loader, val_loader, model, device, eval_iter)
                
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)

                print(f"Epoch {epoch+1}/{num_epochs}, Step {global_step}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Tokens Seen: {tokens_seen}")

        # one sample per epoch, after every training batch
        # step 8: generate sample text for visual inspection
        generate_text(
            top_k = 25,
            temperature = 0.5,
            model = model,
            max_new_tokens = 200, 
            start_context = "Every effort moves you",
            tokenizer = tokenizer, 
            device = device
        )        
        model.train()

    return train_losses, val_losses, track_tokens_seen

            

In [38]:
device =  'mps' if torch.backends.mps.is_available() else 'cpu'

train_losses, val_losses, track_tokens_seen = train_model(
    model, 
    optimizer, 
    train_loader, 
    val_loader, 
    num_epochs= 10, 
    device=device, 
    eval_freq=5, 
    eval_iter=5, 
    start_context="Every effort moves you", 
    tokenizer=tokenizer)

Epoch 1/10, Step 5, Train Loss: 0.0073, Val Loss: 7.1529, Tokens Seen: 2560
Epoch 2/10, Step 10, Train Loss: 0.0065, Val Loss: 7.1543, Tokens Seen: 5120
Epoch 2/10, Step 15, Train Loss: 0.0101, Val Loss: 7.1625, Tokens Seen: 7680
Epoch 3/10, Step 20, Train Loss: 0.0066, Val Loss: 7.1484, Tokens Seen: 10240
Epoch 3/10, Step 25, Train Loss: 0.0072, Val Loss: 7.1822, Tokens Seen: 12800
Epoch 4/10, Step 30, Train Loss: 0.0061, Val Loss: 7.1506, Tokens Seen: 15360
Epoch 4/10, Step 35, Train Loss: 0.0067, Val Loss: 7.1589, Tokens Seen: 17920
Epoch 5/10, Step 40, Train Loss: 0.0059, Val Loss: 7.1524, Tokens Seen: 20480
Epoch 5/10, Step 45, Train Loss: 0.0068, Val Loss: 7.1633, Tokens Seen: 23040
Epoch 6/10, Step 50, Train Loss: 0.0077, Val Loss: 7.1638, Tokens Seen: 25600
Epoch 7/10, Step 55, Train Loss: 0.0066, Val Loss: 7.1764, Tokens Seen: 28160
Epoch 7/10, Step 60, Train Loss: 0.0068, Val Loss: 7.1737, Tokens Seen: 30720
Epoch 8/10, Step 65, Train Loss: 0.0054, Val Loss: 7.1815, Tokens Se

In [40]:
tokenizer =  tiktoken.get_encoding("gpt2")

generated_text =  generate_text(
    top_k = 25,
    temperature = 0.5,
    model = model,
    max_new_tokens = 200, 
    start_context = "Every effort moves you",
    tokenizer = tokenizer, 
    device = device
)
print(f"Output text: \n {generated_text}")

Output text: 
 Every effort moves you?"

"Yes--quite insensible to the irony. She wanted him vindicated--and by me!"

He laughed again, and threw back his head to look up at the sketch of the donkey. "There were days when I couldn't look at that thing--couldn't face it. But I forced myself to put it here; and now it's cured me--cured me. That's the reason why I don't dabble any more, my dear Rickham; or rather Stroud himself is the reason."

For the first time my idle curiosity about my companion turned into a serious desire to understand him better.

"I wish you'd tell me how it happened," I said.

He stood looking up at the sketch, and twirling between his fingers a cigarette he had forgotten to light. Suddenly he turned toward me.

"I'd rather like to tell you--because I've always suspected you of loathing my work
